# Main Radars, Brand, Social, Culture Converts

In [ ]:
import io
import time
import random
import json
import re
import pandas as pd
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
import google.genai as genai
from google.cloud import storage, bigquery, secretmanager
from google.api_core import exceptions

# Configuration

In [ ]:
PROJECT_ID = 'converged-brandradar-poc'
DATASET_ID = "brandradar"
REGION = 'europe-west2'
SECRET_ID = "GEMINI_API_KEY"

# Model and Parameters
# MODELNAME = 'models/gemini-3-pro-preview'
MODELNAME = 'models/gemini-3.1-pro-preview'
# MODELNAME = 'models/gemini-2.5-pro'
BRAND = True
SOCIAL = True
CULTURE = True
REPORTNAME = 'CVS Pharmacy'
DATASOURCE = 'gs://converged-brandradar/lookups/cvs.csv'

# Templates
TEMPLATES = {
    'Brand': 'gs://converged-brandradar/templates/BrandRadar_Template v2.xlsx',
    'Social': 'gs://converged-brandradar/templates/SocialRadar_Template v2.xlsx',
    'Culture': 'gs://converged-brandradar/templates/CultureConverts_template.xlsx'
}

# Utilities

In [ ]:
def access_secret_version(project_id: str, secret_id: str, version_id: str = "latest") -> str:
    """Accesses the payload for the given secret version."""
    client = secretmanager.SecretManagerServiceClient()
    name = f"projects/{project_id}/secrets/{secret_id}/versions/{version_id}"
    try:
        response = client.access_secret_version(request={"name": name})
        return response.payload.data.decode("UTF-8")
    except Exception as e:
        print(f"Error accessing secret: {e}")
        return None


In [ ]:
def extract_json_robust(response_text):
    """Robust JSON extraction with multiple fallback strategies."""
    if not response_text:
        return None

    for pattern in [r'```json\s*(\{.*?\})\s*```', r'(\{.*\})']:
        match = re.search(pattern, response_text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(1))
            except json.JSONDecodeError:
                pass

    try:
        return json.loads(response_text)
    except json.JSONDecodeError:
        return None

In [ ]:
def retry_with_backoff(retries=5, backoff_in_seconds=1):
    def decorator(f):
        def wrapper(*args, **kwargs):
            _retries, _backoff = retries, backoff_in_seconds
            while _retries > 1:
                try:
                    return f(*args, **kwargs)
                except exceptions.ResourceExhausted as e:
                    sleep = _backoff + random.uniform(0, 1)
                    print(f"API quota exceeded. Retrying in {sleep:.2f}s...")
                    time.sleep(sleep)
                    _backoff *= 2
                    _retries -= 1
                except Exception as e:
                    print(f"An unexpected error occurred: {e}. Retrying...")
                    time.sleep(_backoff)
                    _retries -= 1
            return f(*args, **kwargs)
        return wrapper
    return decorator

In [ ]:
def get_question(task_name, brand, category, market):
    if task_name == 'Brand':
        return f"""Please provide a brief (approximately 200 words) summary of the {brand} brand's perception in the {category} category in the {market} market.

          For each of the following aspects, provide a score out of a 100 with a very short description of the score:
        - How often is it mentioned?
        - What is your overall impression of the brand?
        - Have people purchased the brand?
        - Is it thought of as a premium brand that people would pay a premium price for?
        - It is considered a sustainable brand?
        - do people trust it?
        - Would they recommend it to others?
        - What do you think of it's speed, dynamism and willingness to adapt to change?
        - Does the brand meet functional requirements, for example, seamless digital, in-store or purchasing experience, product range, highly innovative,
        value for money, offers price consistency, time saving, high quality products, respectful of customers and their data, offers exclusive experiences,
        offers unique products and services, safe products and services, acts like a leader or challenger brand, has a good reputation, delivers on what it says?        - Does the brand meet personal requirements, for example, image enhancing, personalised offers, confidence building, health enhancing, lets me escape from the everyday, inspires me with new opportunities and ideas,
        - Does the brand meet personal requirements, for example, image enhancing, personalised offers, confidence building, health enhancing,
        lets me escape from the everyday, inspires me with new opportunities and ideas, instilling peace of mind and confidence, connecting people,
        enables me to be smart with money and time, simplifies my life, helps me express myself as an individual, feel more attractive and stylish,
        make me feel special and unique, makes me energised and alive, helps me feel in control of my life, gives me a sense of happiness, makes me feel good,
        makes me feel like it is looking after my best interests?
         - Does the brand meet collective requirements, for example does it act with integrity, is it ethical, is it transparent in its operations,
         is it considered a good employer, offers sustainable products, it fights against poverty and hunger, supports healthy living and promotes well-being,
         supports culture and education, supports equality, diversity and inclusion, supports social issues and good causes,
         contributes to the national economy and my local community, invests in innovative, sustainable and ethical solutions, inspires sustainable,
         responsible behaviours and consumption, committed to making products/services more sustainable?

        and can you provide short descriptions of the following questions
        - What are the most common touchpoints associated with the brand?
        - Which channel do consumers mention most when discussing the brand?
        - Where has the brand appeared most in the consumer journey online
        - What formats of branded content does brand most often use?
        - What recent sponsorships or partnerships has the brand activated?
        - What kinds of consumer-generated content appear in discussions about the brand?
        - What kinds of brand-generated content appear in discussions about the brand?



        Could the results be returned in JSON format with the brand in a variable called brand, the country in a field called market, the summary in a field called summary,
        the individual scores and the short descriptions of the scores in fields called, mentions, mentions_description, overallimp, overallimp_description, purchase, purchase_description, premium, premium_description,
        sustainable, sustainable_description, trust, trust_description, recommend, recommend_description, dynamic, dynamic_description, functional, functional_description, personal, personal_description, collective, collective_description,
        touchpoints_description, channel_description, journey_description, branded_description, sponsorship_description, cgc_description, brand_generated_description?
  """

    elif task_name == 'Social':
        return f"""Please provide a brief (approximately 200 words) summary of the {brand} brand’s public sentiment and online conversation dynamics in the {category} category in the {market} market based on recent digital signals.

        	For each of the following aspects, provide a score out of 100 and a short description explaining the score:
            - What is the general sentiment about the brand?
            - What emotions are most commonly associated with the brand?
            - Is the brand currently viewed more positively overall?
            - Is the brand currently viewed more negatively overall?
            - Are there noticeable sentiment shifts over the last 6 months?
            - Have there been sentiment spikes triggered by campaigns, news, or events?
            - What are the most discussed themes in online conversations about the brand?
            - What social, cultural, product or service topics are emerging?
            - Are there any polarizing or divisive conversations surrounding the brand?
            - Which platforms (e.g. Reddit, X, TikTok, forums, blogs) host the most conversations about the brand?
            - Where does the brand get the most engagement?
            - Is brand visibility driven more by owned channels or third-party/earned media?

          Please return results in **JSON format** the using the following fields:
          - `"brand"` for the brand name
          - `"market"` for the country
          - `"summary"` for the narrative overview
          - `"scores"` as an object with:
            - `sentiment`, `emotion`, `positivity`, `sentiment_shift`, `sentiment_spike`
            - `themes`, `emerging_topics`, `debates`
            - `platforms`, `engagement`, `media_type`

          Each object should contain:
          - `score` (0–100)
          - `description` (1–2 sentence explanation)"""

    elif task_name == 'Culture':

        return f"""Please analyze the {brand} brand's perception in the {category} category in the {market} market.
            I am interested in understanding 4 areas, buzz, belonging, belief and behaviour. Could provide a value between 1 and 100 for the following questions and store them in JSON format,
            can you provide a brief (approximately 50-100 words) summary of the reasons behind these scores?

            Buzz1: Measuring attention. Number of mentions (total #) - across social, news, forums.
            Buzz2: Share of cultural voice. % of culturally relevant mentions vs peer brands in the target market.
            Buzz3:  Competitor share. Share of voice vs competitors (%) - brand mentions as a % of category mentions.
            Buzz4: Authority-weighted presence. Weighted score based on coverage in high-authority media and top creators.
            Buzz5: Topic spread. Count of distinct cultural themes (e.g., sustainability, arts, finance) where the brand appears organically.
            Belong1: Cultural fit score. % of contextual mentions coded as “authentic” vs “performative”.
            Belong2: Community acceptance index. Proportion of mentions coming from identified target communities/subcultures (%).
            Belong3: Peer comparison on affinity. Relative score vs. category peers for perceived cultural fit.
            Belong4: Audience inclusion signal. Frequency of language indicating inclusion (e.g., “for us,” “represents,” “part of”, “this is for people like”).
            Belong5: Negative fit incidence. % of mentions expressing rejection or misalignment with cultural spaces.
            Belief1: Cultural authority score. % of mentions framing the brand as a leader, innovator, or credible voice in cultural or social topics.
            Belief2: Value alignment index. Frequency of positive associations with values (e.g., sustainability, diversity, ethics) in cultural discourse.
            Belief3: Thought leadership presence. Number of named endorsements or quotes from recognised cultural leaders/experts.
            Belief4: Peer Comparison on Authority. Relative score vs category peers for perceived leadership and credibility.
            Belief5: Negative Authority Incidence. % of mentions questioning or criticizing the brand’s intent or values.
            Behave1: Action conversation signals. Frequency of mentions indicating concrete action (e.g., “I attended”, “I bought”, “I downloaded”, purchase, trial, sign-up).
            Behave2: Advocacy rate. % of user-generated content showing active recommendation or endorsement of the brand.
            Behave3: Cultural participation index. Number of instances where audiences join brand-led cultural initiatives (e.g., events, challenges, collaborations).
            Behave4: Behavioural shift mentions. Mentions where users report changing habits or choices influenced by the brand’s cultural activity.
            Behave5: Peer comparison on adoption. Relative score vs category peers for observable cultural-driven actions

            These should be provided in JSON
            format with fields: Brand, Category, Market, Buzz1, Buzz2, Buzz3, Buzz4, Buzz5, Belong1, Belong2, Belong3,
            Belong4, Belong5, Belief1, Belief2, Belief3, Belief4, Belief5, Behave1, Behave2, Behave3, Behave4, Behave5,
            Buzz1Desc, Buzz2Desc, Buzz3Desc, Buzz4Desc, Buzz5Desc, Belong1Desc, Belong2Desc, Belong3Desc, Belong4Desc,
            Belong5Desc, Belief1Desc, Belief2Desc, Belief3Desc, Belief4Desc, Belief5Desc, Behave1Desc, Behave2Desc,
            Behave3Desc, Behave4Desc, Behave5Desc,
            date, projectname

            CRITICAL: Return ONLY a valid JSON object with NO markdown formatting, NO additional text."""

    else:
        raise ValueError(f"Invalid task name: {task_name}")





# State and Schema Definitions

In [ ]:
ANALYSIS_CONFIGS = {
    'Brand': {
        'run': BRAND,
        'table_id': 'BrandResults',
        'sheet': 'BrandRadar Results',
        'template': TEMPLATES['Brand'],
        'int_cols': ["mentions", "overallimp", "purchase", "premium", "sustainable",
                     "trust", "recommend", "dynamic", "functional", "personal", "collective"],
        'str_cols': ["touchpoints", "channel", "journey", "branded", "sponsorship", "cgc", "brand_generated"]
    },
    'Social': {
        'run': SOCIAL,
        'table_id': 'SocialResults',
        'sheet': 'SocialRadar Results',
        'template': TEMPLATES['Social'],
        'int_cols': ["sentiment", "emotion", "positivity", "shift", "spike", "themes",
                     "emerging_topics", "debates", "platforms", "engagement", "media_type"],
        'str_cols': []
    },
    'Culture': {
        'run': CULTURE,
        'table_id': 'CultureConverts',
        'sheet': 'Cultural Converts',
        'template': TEMPLATES['Culture'],
        'int_cols': [f"Buzz{i}" for i in range(1, 6)] + [f"Belong{i}" for i in range(1, 6)] + \
                    [f"Belief{i}" for i in range(1, 6)] + [f"Behave{i}" for i in range(1, 6)],
        'str_cols': [f"Buzz{i}Desc" for i in range(1, 6)] + [f"Belong{i}Desc" for i in range(1, 6)] + \
                    [f"Belief{i}Desc" for i in range(1, 6)] + [f"Behave{i}Desc" for i in range(1, 6)]
    }
}

# Attach full columns list for dataframe initialization mapping
for t, c in ANALYSIS_CONFIGS.items():
    if t == 'Culture':
        c['columns'] = ["Brand", "Category", "Market"] + c['int_cols'] + c['str_cols'] + \
        ["date", "projectname"]
    else:
        c['columns'] = ["Brand", "Category", "Market", "Summary"] + \
                       c['int_cols'] + \
                       [f"{col}_description" for col in c['int_cols']] + \
                       [f"{col}_description" for col in c['str_cols']] + \
                       ["date"]


In [ ]:
def initialize_dataframe(task_name):
    """Dynamically set up DataFrame based on schema."""
    cols = ANALYSIS_CONFIGS[task_name]['columns']
    df = pd.DataFrame(columns=cols)
    for col in ANALYSIS_CONFIGS[task_name]['int_cols']:
        df[col] = pd.Series(dtype='int64')
    df['date'] = pd.Series(dtype='datetime64[ns]')
    return df


# API Logic

In [ ]:
@retry_with_backoff(retries=3, backoff_in_seconds=2)
def run_query(client, question):
    try:
        response = client.models.generate_content(model=MODELNAME, contents=question)
        if not response.candidates:
            print(f"Request blocked. Feedback: {response.prompt_feedback}")
            return None
        return response.text
    except ValueError:
        print("Response empty or blocked.")
        return None


In [ ]:
def extract_values_to_row(task_name, json_data, row_template):
    """Parses logic correctly depending on specific task expectations."""
    if not json_data:
        return row_template

    config = ANALYSIS_CONFIGS[task_name]
    new_row = row_template.copy()

    # Generic mappings
    if task_name != 'Culture':
        new_row['Summary'] = json_data.get('summary', '')
    #  new_row['Summary'] = json_data.get('summary', '')

    if task_name == 'Social':
        scores = json_data.get('scores', {})
        for measure in config['int_cols']:
            # Adjust the key naming difference specific for social
            key = "sentiment_shift" if measure == "shift" else "sentiment_spike" if measure == "spike" else measure
            if key in scores:
                new_row[measure] = scores[key].get('score', 0)
                new_row[f"{measure}_description"] = scores[key].get('description', '')
    else:
        for k, v in json_data.items():
            if k in new_row or k not in config['columns']: continue
            new_row[k] = v

    return new_row


In [ ]:
def process_record(client, row, task_name):
    """Processes a single brand query."""
    question = get_question(task_name, row['brand'], row['category'], row['market'])
    response_text = run_query(client, question)

    brand_json = extract_json_robust(response_text) if response_text else None

    # Init base row structure
    row_template = {
        "Brand": brand_json.get('brand', row['brand']) if brand_json else row['brand'],
        "Category": row['category'],
        "Market": row['market'],
        "date": datetime.now().strftime("%Y/%m/%d")
    }
    if task_name == 'Culture':
        row_template["projectname"] = REPORTNAME

    row_data = extract_values_to_row(task_name, brand_json, row_template)

    return row_data, question, response_text, bool(brand_json)


In [ ]:
def execute_task(task_name, df_input, client, storage_client):
    """Executes the full pipeline for a specific reporting mode."""
    print(f"\n{'='*40}\nProcessing Task: {task_name}\n{'='*40}")

    df = initialize_dataframe(task_name)
    all_responses = []

    for idx, row in df_input.iterrows():
        print(f"[{idx+1}/{len(df_input)}] Processing {row['brand']} ({row['market']})")
        row_data, question, resp_text, success = process_record(client, row, task_name)

        df = pd.concat([df, pd.DataFrame([row_data])], ignore_index=True)

        all_responses.append({
            "timestamp": datetime.now().isoformat(),
            "brand": row['brand'],
            "market": row['market'],
            "input_question": question,
            "gemini_response": resp_text,
            "success": success
        })
        time.sleep(1) # Rate limiter
        # print(resp_text)

    df['projectname'] = REPORTNAME

    # --- Add Calculated Fields for Culture ---
    if task_name == 'Culture':
        print("Adding calculated fields for Culture task...")

        df['Buzzscore'] = (df['Buzz1'] + df['Buzz2'] + df['Buzz3'] + df['Buzz4'] + df['Buzz5']) / 5

        df['Belongscore'] = (df['Belong1'] + df['Belong2'] + df['Belong3'] + df['Belong4'] + (100 - df['Belong5'])) / 5

        df['Beliefscore'] = (df['Belief1'] + df['Belief2'] + df['Belief3'] + df['Belief4'] + (100 - df['Belief5'])) / 5

        df['Behavescore'] = (df['Behave1'] + df['Behave2'] + df['Behave3'] + df['Behave4'] + df['Behave5']) / 5

        df['CultureScore'] = (
            df['Buzz1'] + df['Buzz2'] + df['Buzz3'] + df['Buzz4'] + df['Buzz5'] +
            df['Belong1'] + df['Belong2'] + df['Belong3'] + df['Belong4'] + (100 - df['Belong5']) +
            df['Belief1'] + df['Belief2'] + df['Belief3'] + df['Belief4'] + (100 - df['Belief5']) +
            df['Behave1'] + df['Behave2'] + df['Behave3'] + df['Behave4'] + df['Behave5']
        ) / 20
    # -----------------------------------------

    # Exporters
    bucket = storage_client.bucket("converged-brandradar")

    # 1. JSON
    blob_name = f"results/{task_name} {datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    bucket.blob(blob_name).upload_from_string(json.dumps(all_responses, indent=4), content_type='application/json')
    print(f"JSON Exported: gs://converged-brandradar/{blob_name}")

    # 2. XLS
    config = ANALYSIS_CONFIGS[task_name]
    xls_path = f"gs://converged-brandradar/results/{task_name} {REPORTNAME.strip()}.xlsx"

    template_bucket = storage_client.bucket(config['template'].split('/')[2])
    template_blob = template_bucket.blob('/'.join(config['template'].split('/')[3:]))
    template_buffer = io.BytesIO()
    template_blob.download_to_file(template_buffer)
    template_buffer.seek(0)

    book = load_workbook(template_buffer)
    book['Key'].cell(row=3, column=1, value=REPORTNAME)
    sheet = book[config['sheet']]

    for r_idx, out_row in enumerate(dataframe_to_rows(df, index=False, header=False), 2):
        for c_idx, value in enumerate(out_row, 1):
            sheet.cell(row=r_idx, column=c_idx, value=value)

    out_buffer = io.BytesIO()
    book.save(out_buffer)
    out_buffer.seek(0)

    out_bucket = storage_client.bucket(xls_path.split('/')[2])
    out_bucket.blob('/'.join(xls_path.split('/')[3:])).upload_from_file(out_buffer, content_type='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')
    print(f"Excel Exported: {xls_path}")

    # 3. BigQuery
    table_ref = f"{PROJECT_ID}.{DATASET_ID}.{config['table_id']}"
    schema = []
    for col in df.columns:
        dtype = df[col].dtype
        bq_type = "INTEGER" if dtype == 'int64' else "FLOAT" if dtype == 'float64' else "TIMESTAMP" if dtype == 'datetime64[ns]' else "STRING"
        schema.append(bigquery.SchemaField(col, bq_type))

    bq_client = bigquery.Client(project=PROJECT_ID)
    try:
        bq_client.get_table(table_ref)
        job_config = bigquery.LoadJobConfig(write_disposition=bigquery.WriteDisposition.WRITE_APPEND)
    except Exception:
        job_config = bigquery.LoadJobConfig(write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE, schema=schema)

    bq_client.load_table_from_dataframe(df, table_ref, job_config=job_config).result()
    print(f"BigQuery Updated: {config['table_id']} ({len(df)} rows)")


# Entrypoint

In [ ]:
def main():
    start_time = time.time()

    api_key = access_secret_version(PROJECT_ID, SECRET_ID)
    if not api_key:
        print("Failed to retrieve API key!")
        return

    client = genai.Client(api_key=api_key)
    storage_client = storage.Client()

    # Load and process Data
    dfbrands = pd.read_csv(DATASOURCE)
    # dfbrands = dfbrands.head(2) # Drop head() modifier when ready
    # dfbrands = dfbrands.iloc[5:] # Drop a number of records if rerunning or splitting
    cats = dfbrands.groupby(['market', 'category']).size().reset_index()
    cats['brand'] = 'All Category'
    dfbrands = pd.concat([dfbrands, cats], ignore_index=True)

    for task_name, config in ANALYSIS_CONFIGS.items():
        if config['run']:
            execute_task(task_name, dfbrands, client, storage_client)

    print(f"\nTotal Run time: {time.time() - start_time:.2f} seconds")
    # Remove Category totals for Culture
    # for m in client.models.list():
    #     print(m.name)



In [ ]:
if __name__ == "__main__":
    main()


Processing Task: Brand
[1/17] Processing CVS Health (USA)
[2/17] Processing Walgreens Pharmacy (USA)
[3/17] Processing Amazon Pharmacy (USA)
[4/17] Processing Walmart Pharmacy (USA)
[5/17] Processing Costco Pharmacy (USA)
[6/17] Processing OptumRx (USA)
[7/17] Processing Express Scripts (USA)
[8/17] Processing Prime Therapeutics (USA)
[9/17] Processing Humana Pharmacy Solutions (USA)
[10/17] Processing Capital RX (USA)
[11/17] Processing Capsule Pharmacy (USA)
[12/17] Processing Kroger Health (USA)
[13/17] Processing Alto Pharmacy (USA)
[14/17] Processing NowRx (USA)
[15/17] Processing Nurx (USA)
[16/17] Processing Ro (Roman) (USA)
[17/17] Processing All Category (USA)
JSON Exported: gs://converged-brandradar/results/Brand 20260412_181538.json
Excel Exported: gs://converged-brandradar/results/Brand CVS Pharmacy.xlsx
BigQuery Updated: BrandResults (17 rows)

Processing Task: Social
[1/17] Processing CVS Health (USA)
[2/17] Processing Walgreens Pharmacy (USA)
[3/17] Processing Amazon Ph

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/gemini-embedding-001
models/aqa
models/imagen-4.0-generate-001
models/imagen-4.0-ultra-generate-001
models/imagen-4.0-fast-generate-001
models/veo-2.0-generate-001
models/veo-3.0-generate-001
models/veo-3.0-fast-generate-001
models/veo-3.1-generate-preview
models/veo-3.1-fast-generate-preview
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
